# Bookstore Assistant — How This Project Works

This notebook builds a **chatbot that can look up real book prices** (buy, sell, and rent) using the BooksRun API, and lets an LLM (DeepSeek) decide *when* to call that API on its own.

The core idea being practiced here is **tool use / function calling**:

1. We write normal Python functions that do something useful (look up an ISBN, get a price).
2. We describe each function to the LLM in a special JSON format so the model knows the function exists, what it does, and what arguments it needs.
3. When a user asks a question, the LLM decides for itself whether it needs to call one of these functions to answer properly — it doesn't just guess prices, it fetches real data.
4. We run the function on the model's behalf, send the result back to the model, and let it continue until it has a final answer.

The notebook is organized like this:
1. **Setup** — imports and API keys
2. **Tools** — the actual Python functions that talk to BooksRun / Open Library
3. **Tool schemas** — JSON descriptions of those functions for the LLM
4. **The assistant logic** — system prompt, the tool-calling loop, and the chat function
5. **Putting it together** — a Gradio chat UI, and a plain script test of the tools

---
### Setup
The cell below imports everything the project needs:
- `os`, `dotenv` — to load secret API keys from a `.env` file instead of hardcoding them
- `openai` (the `OpenAI` client) — used here to talk to the **DeepSeek** API, since DeepSeek's API is OpenAI-compatible
- `gradio` — to build a simple web chat interface at the end
- `IPython.display` — to render the assistant's markdown-formatted replies nicely in the notebook


In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import display, Markdown




### Connecting to the DeepSeek API
This cell loads the DeepSeek API key from the `.env` file and creates a client to talk to DeepSeek.

- `load_dotenv()` reads a local `.env` file and makes its variables available via `os.getenv(...)`.
- We check the key exists (and only print the first few characters, so we never leak the full secret).
- `MODEL` is the name of the DeepSeek model we'll use for chat completions.
- `deepseek = OpenAI(base_url=..., api_key=...)` creates a client pointed at DeepSeek's servers, but using the familiar OpenAI SDK — this works because DeepSeek's API follows the same format as OpenAI's.


In [ ]:
load_dotenv(override=True)

deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
if deepseek_api_key:
    print(f"Deepseek API Key exists and begins {deepseek_api_key[:8]}")
else:
    print("Deepseek API Key not set")
    
MODEL = "deepseek-chat"
deepseek_url = "https://api.deepseek.com"
deepseek = OpenAI(base_url=deepseek_url, api_key=deepseek_api_key)

## Step 1: Writing the actual tools

Before an LLM can "use a tool," we need real Python code that does the work. This section defines the functions that talk to the **BooksRun** API (for prices) and the **Open Library** API (for finding a book's ISBN).

### Looking up a book's ISBN
Most price APIs need a precise identifier (an ISBN) rather than a fuzzy title. So the first tool, `find_isbn`, takes a book title (and optionally an author) and:
1. Searches the free, keyless **Open Library** search API for matching books
2. Picks the best ISBN found (prefers a 13-digit ISBN if available)
3. Returns that ISBN as a string, or raises an error if nothing was found

This means the assistant can accept natural requests like *"how much for the 48 laws of power?"* without the user ever typing an ISBN themselves.

*(Note: `API_KEY` here is your personal BooksRun key — in a real project this should also live in the `.env` file rather than being hardcoded.)*


In [ ]:
"""
Get book prices from the BooksRun API by title (auto-looks up the ISBN).

Requires: pip install requests
Get your BooksRun API key by signing up at https://booksrun.com/
"""

import requests

API_KEY = "yxma23gx51mo6t7lh8do"  # <-- put your BooksRun API key here

BUYBACK_URL = "https://booksrun.com/api/price/sell/{isbn}"
BUY_RENT_URL = "https://booksrun.com/api/v3/price/buy/{isbn}"
OPEN_LIBRARY_SEARCH_URL = "https://openlibrary.org/search.json"


def find_isbn(title: str, author: str = None) -> str:
    """
    Look up a book by title (and optional author) using the free
    Open Library search API, and return the first ISBN found.
    Raises ValueError if nothing is found.
    """
    query = title if not author else f"{title} {author}"
    response = requests.get(
        OPEN_LIBRARY_SEARCH_URL,
        params={
            "q": query,
            "limit": 5,
            "fields": "title,author_name,isbn",  # ensures isbn is included in results
        },
    )
    response.raise_for_status()
    docs = response.json().get("docs", [])
 
    for doc in docs:
        isbns = doc.get("isbn", [])
        isbn_13 = next((i for i in isbns if len(i) == 13), None)
        chosen = isbn_13 or (isbns[0] if isbns else None)
        if chosen:
            authors = ", ".join(doc.get("author_name", ["?"]))
            print(f"  Found: \"{doc.get('title')}\" by {authors} (ISBN {chosen})")
            return f"{chosen}"
 
    raise ValueError(f"No ISBN found for '{query}'")
 



### Getting prices from BooksRun
Now that we can turn a title into an ISBN, these two functions call the **BooksRun** API to get actual price data:

- **`get_buyback_price(isbn)`** — asks "how much would BooksRun pay *me* if I sold this book back to them?" (the buyback/sell price).
- **`get_buy_rent_price(isbn)`** — asks "what are my options if I want to *buy* or *rent* this book?" (new/used prices, rental terms, marketplace sellers, etc).

Both functions call the BooksRun endpoint with the ISBN, check whether BooksRun returned an error, and return a plain-text summary the LLM can read and pass along to the user.


In [ ]:
def get_buyback_price(isbn: str) -> dict:
    """
    Get the price BooksRun will pay you for a book (buyback/sell price).
    Returns a dict like {"Average": 7.91, "Good": 8.47, "New": 8.75}
    or raises ValueError on error.
    """
    url = BUYBACK_URL.format(isbn=isbn)
    response = requests.get(url, params={"key": API_KEY})
    response.raise_for_status()
    data = response.json()["result"]
 
    if data["status"] == "error":
        raise ValueError(f"BooksRun error: {data['text']}")
 
    return f"the details of the book are {data["text"]}"  # dict of condition -> price
 
 
def get_buy_rent_price(isbn: str) -> dict:
    """
    Get prices for buying/renting a book (new, used, rental terms, etc).
    Returns the full 'offers' dict from the API.
    """
    
    url = BUY_RENT_URL.format(isbn=isbn)
    
    response = requests.get(url, params={"key": API_KEY})
    response.raise_for_status()
    data = response.json()["result"]
 
    if data["status"] == "error":
        raise ValueError(f"BooksRun error: {data['message']}")
 
    return f"here are the details of the book with isbn: {isbn}\n\n{data["offers"]}"


## Step 2: Describing the tools to the LLM

An LLM can't see our Python code — it can only read a **JSON description (schema)** of each function: its name, what it does, and what arguments it expects. This is what "function calling" / "tool use" means in practice.

Each schema below follows the format required by the API:
- `name` — must exactly match the Python function name
- `description` — tells the model *when* and *why* to use this tool (the model reads this to decide!)
- `parameters` — a JSON Schema describing the expected arguments, which ones are required, etc.

Below is the schema for `find_isbn` — the model will call this whenever it needs to convert a book title into an ISBN.


In [ ]:
get_isbn_function = {
    "name": "find_isbn",
    "description": """Look up a book by title (and optional author) using the free
    Open Library search API, and return the first ISBN found.
    Raises ValueError if nothing is found.""",
    
    "parameters": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "The title of the book the customer wants to buy/sell/rent",
            }, 
            "author": {
                "type": "string",
                "description": "The author of the book the customer wants to buy/sell.rent"
            }
        },
        "required": ["title"],
        "additionalProperties": False
    }
}

Here's the schema for `get_buyback_price` — the model calls this once it already has an ISBN and the customer wants to **sell** a book.


In [ ]:
# There's a particular dictionary structure that's required to describe our function:

buyback_function = {
    "name": "get_buyback_price",
    "description": """Get the price BooksRun will pay you for a book (buyback/sell price).
    Returns a dict like {"Average": 7.91, "Good": 8.47, "New": 8.75}
    or raises ValueError on error.""",
    
    "parameters": {
        "type": "object",
        "properties": {
            "isbn": {
                "type": "string",
                "description": "The isbn of the book the seller wants to sell",
            },
        },
        "required": ["isbn"],
        "additionalProperties": False
    }
}

And here's the schema for `get_buy_rent_price` — the model calls this once it has an ISBN and the customer wants to **buy or rent** a book.


In [ ]:
# There's a particular dictionary structure that's required to describe our function:

buy_rent_function = {
    "name": "get_buy_rent_price",
    "description": """Get prices for buying/renting a book (new, used, rental terms, etc).
    Returns the full 'offers' dict from the API.""",
    
    "parameters": {
        "type": "object",
        "properties": {
            "isbn": {
                "type": "string",
                "description": "The isbn of the book the buyer/rentee wants to buy/rent",
            },
        },
        "required": ["isbn"],
        "additionalProperties": False
    }
}

## Step 3: Bundling the tools together

The API expects one combined list of tools, where each entry is wrapped as `{"type": "function", "function": <schema>}`. This `tools` list is what actually gets passed to the DeepSeek API on every request, so the model always knows all three tools are available to it.


In [ ]:
tools = [{"type": "function", "function": buyback_function},
{"type": "function", "function": buy_rent_function},
{"type": "function", "function": get_isbn_function}]
tools

## Step 4: Giving the assistant a personality (system prompt)

The **system prompt** sets the ground rules and personality for the assistant before any conversation starts. Here it's told:
- who it works for and what it can help with (buying/selling/renting books)
- that it must have an ISBN before it can fetch prices (nudging it to use `find_isbn` first)
- to keep answers short, funny, and snarky
- to be honest if a book isn't available, rather than inventing information
- to respond using markdown formatting

This is the model's "instructions," not something the user sees directly.


In [ ]:
system_prompt = """You are a bookstore assistant working for sigma books, you a bookstore. You help customers buy, rent, or sell books.
You can only get the price details of a book if you have the isbn number of that book. Give short funny
snarky answers. If the book the customer wants is not available, say so, do not make up any unwanted information. Make sure
the customer leaves the store satisfied. Respond in markdown"""

## Step 5: Actually running the tools (the dispatcher)

When the LLM decides it wants to use a tool, it doesn't run any code itself — it just replies with a *request* to call a specific function with specific arguments. It's our job to actually execute that function.

`handle_tool_call(message)` does exactly that:
1. Reads the list of tool calls the model asked for (`message.tool_calls`)
2. For each one, checks which function was requested (`get_buy_rent_price`, `get_buyback_price`, or `find_isbn`)
3. Parses the arguments the model provided (they arrive as a JSON string, so `json.loads` turns them into a Python dict)
4. Calls the *real* Python function with those arguments
5. Packages the result back into the special `{"role": "tool", ...}` message format the model expects, tagged with the same `tool_call_id` so the model knows which request this result answers

The list of these tool results gets sent back to the model so it can continue the conversation with real data in hand.


In [ ]:
def handle_tool_call(message):
    tool_calls = message.tool_calls
    print(message.tool_calls)
    # print(type(message.tool_calls))
    # print(type(tool_call))
    updates = []
    for tool_call in tool_calls:
        if tool_call.function.name == "get_buy_rent_price":
            arguments = json.loads(tool_call.function.arguments)
            book = arguments.get('isbn')
            price_details = get_buy_rent_price(book)
            response = {
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            }
            updates.append(response)
        
        elif tool_call.function.name == "get_buyback_price":
            arguments = json.loads(tool_call.function.arguments)
            book = arguments.get('isbn')
            price_details = get_buyback_price(book)
            response = {
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            }
            updates.append(response)
        elif tool_call.function.name == "find_isbn":
            arguments = json.loads(tool_call.function.arguments)
            book = [arguments.get('title'), arguments.get('author', None)]
            isbn_value = find_isbn(title=book[0])
            print(isbn_value)
            response = {
                "role": "tool",
                "content": isbn_value,
                "tool_call_id": tool_call.id 
            }
            updates.append(response)
            print(response)

    return updates

## Step 6: Wrapping the loop in a reusable `chat()` function

This function's signature (`message, history`) matches what Gradio's `ChatInterface` expects, which is why it can be plugged straight into the UI in the next cell.


In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = deepseek.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response = deepseek.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

## Step 7: Launching the chat UI

`gr.ChatInterface(fn=chat, type="messages").launch()` spins up a simple local web page where you can chat with the bookstore assistant interactively. Gradio calls our `chat()` function every time you send a message, passing in the conversation so far — everything we built above (tool schemas, the dispatcher, the loop) runs behind the scenes for every reply.


In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()